# 05. SurvFace 공식 평가 및 시각화

04의 top-20 distance 결과로 SurvFace MATLAB 절차와 같은 방향의 `TPIR20@FPIR=0.1/0.2/0.3` 및 TPIR–FPIR AUC를 계산합니다. 단, 검색 거리는 원본 코드의 Euclidean이 아니라 pgvector cosine distance이므로 결과 이름에 adaptation을 명시합니다.

| 모드 | 예상 시간 |
| --- | ---: |
| `EXECUTE_STAGE=False` | 1초 미만 |
| 전체 CSV 평가·그림 | 약 1~10분 |

> **진행/체크포인트/재시작**: CSV 로드, curve 계산, artifact/figure 저장마다 진행 상태를 출력합니다. 중단되면 Kernel Restart 후 05 전체를 재실행하여 새 attempt를 만듭니다. `FINALIZE_RUN=False`가 기본이므로 profile/mode별 결과를 더 추가할 수 있습니다. 모든 실험을 확인한 뒤에만 True로 바꿉니다.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("D:/ronbun 내부에서 노트북을 실행하십시오.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from research.runtime import ProgressReporter, RunStore, resolve_active_run

EXECUTE_STAGE = False
ALLOW_SMOKE_EVALUATION = False
FINALIZE_RUN = False
CURVE_STEPS = 1000
TARGET_FPIRS = (0.1, 0.2, 0.3)
RUN_ROOT = PROJECT_ROOT / "runs" / "survface"

def resolve_run_for_preflight():
    try:
        return resolve_active_run(
            RUN_ROOT, environment_variable="RONBUN_SURVFACE_RUN_DIR"
        ), None
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"

RUN_DIR, RUN_RESOLUTION_ERROR = resolve_run_for_preflight()
PROGRESS = ProgressReporter("SurvFace 05 official evaluation", heartbeat_seconds=30)


## 1. 평가 입력 preflight

가장 최근 completed 04 attempt만 읽습니다. 기본값에서는 smoke 결과 평가를 거부합니다.


In [ ]:
preflight = {
    "execute_stage": EXECUTE_STAGE,
    "run_dir": str(RUN_DIR) if RUN_DIR else None,
    "run_resolution_error": RUN_RESOLUTION_ERROR,
    "allow_smoke_evaluation": ALLOW_SMOKE_EVALUATION,
    "finalize_run": FINALIZE_RUN,
    "target_fpirs": TARGET_FPIRS,
    "curve_steps": CURVE_STEPS,
}
preflight


## 2. TPIR20–FPIR curve 계산

unmated probe의 top-1 minimum distance로 FPIR threshold를 만들고, mated probe의 true identity가 top 20에 있을 때 그 distance를 사용합니다. top 20에 없으면 `inf`로 두어 identification 실패로 계산합니다. 비교 연산은 공식 MATLAB과 같이 `distance < threshold`입니다.


In [ ]:
def latest_completed_search_csv(run: RunStore) -> Path:
    run.verify_phase_artifacts("04_official_probe_search")
    attempts = run.run_dir / "phases" / "04_official_probe_search" / "attempts"
    completed = []
    for path in sorted(attempts.glob("A*/phase_manifest.json")):
        payload = json.loads(path.read_text(encoding="utf-8"))
        if payload.get("status") == "completed":
            completed.append((int(payload["attempt"]), payload))
    if not completed:
        raise RuntimeError("completed 04 attempt가 없습니다.")
    _, payload = max(completed, key=lambda item: item[0])
    csv_paths = [
        run.run_dir / item["path"]
        for item in payload.get("details", {}).get("artifacts", [])
        if str(item["path"]).lower().endswith(".csv")
    ]
    if len(csv_paths) != 1:
        raise RuntimeError(f"04 search CSV를 하나로 결정할 수 없습니다: {csv_paths}")
    return csv_paths[0]


def official_curve(frame, *, steps: int, rank: int = 20):
    import numpy as np
    import pandas as pd

    registered = frame.loc[frame["probe_type"].eq("registered")]
    unknown = frame.loc[frame["probe_type"].eq("unknown_unknown")]
    if registered.empty or unknown.empty:
        raise ValueError("registered와 unknown_unknown 결과가 모두 필요합니다.")

    negative_distances = np.asarray(
        [float(values[0]) for values in unknown["ranked_distances"]], dtype=float
    )
    minimum = float(np.min(negative_distances))
    maximum = float(np.max(negative_distances))
    thresholds = (
        np.linspace(minimum, maximum, steps + 1)
        if maximum > minimum
        else np.asarray([minimum, np.nextafter(maximum, np.inf)])
    )
    fpirs = np.asarray([(negative_distances < threshold).mean() for threshold in thresholds])

    true_distances = []
    rank20_hits = []
    for row in registered.itertuples(index=False):
        identities = [str(value) for value in row.ranked_identities[:rank]]
        distances = [float(value) for value in row.ranked_distances[:rank]]
        target = str(row.query_identity_id)
        if target in identities:
            position = identities.index(target)
            true_distances.append(distances[position])
            rank20_hits.append(True)
        else:
            true_distances.append(np.inf)
            rank20_hits.append(False)
    true_distances = np.asarray(true_distances, dtype=float)
    tpirs = np.asarray([(true_distances < threshold).mean() for threshold in thresholds])
    order = np.argsort(fpirs, kind="stable")
    curve = pd.DataFrame({
        "threshold": thresholds[order],
        "fpir": fpirs[order],
        "tpir20": tpirs[order],
    })
    metrics = {
        "registered_count": int(len(registered)),
        "unknown_unknown_count": int(len(unknown)),
        "closed_set_rank20": float(np.mean(rank20_hits)),
        "auc_tpir20_fpir": float(np.trapz(curve["tpir20"], curve["fpir"])),
    }
    for target in TARGET_FPIRS:
        index = int(np.argmin(np.abs(curve["fpir"].to_numpy() - target)))
        metrics[f"tpir20_at_fpir_{target:.1f}"] = float(curve.iloc[index]["tpir20"])
        metrics[f"realized_fpir_{target:.1f}"] = float(curve.iloc[index]["fpir"])
        metrics[f"threshold_at_fpir_{target:.1f}"] = float(curve.iloc[index]["threshold"])
    return curve, metrics


result = {"status": "not_executed", **preflight}
if EXECUTE_STAGE:
    if RUN_DIR is None:
        raise RuntimeError(f"SurvFace run을 찾지 못했습니다: {RUN_RESOLUTION_ERROR}")

    import matplotlib.pyplot as plt
    import pandas as pd

    from research.runtime.hashing import sha256_file

    run = RunStore.open(RUN_DIR)
    run.verify_inputs()
    search_path = latest_completed_search_csv(run)
    PROGRESS.emit("04 search CSV 로딩 시작", path=str(search_path))
    search = pd.read_csv(search_path)
    for column in ("ranked_identities", "ranked_distances"):
        search[column] = search[column].map(
            lambda value: json.loads(value) if isinstance(value, str) else value
        )
    if set(search["probe_type"].astype(str)) != {"registered", "unknown_unknown"}:
        raise ValueError("공식 평가는 registered와 unknown_unknown 두 역할만 허용합니다.")
    official_complete = bool(search["official_complete"].astype(bool).all())
    if not official_complete and not ALLOW_SMOKE_EVALUATION:
        raise RuntimeError("04 결과가 full official coverage가 아닙니다. smoke 평가는 기본적으로 거부됩니다.")

    group_columns = ["compression_profile", "search_mode"]
    metric_rows = []
    curve_frames = []
    with run.phase("05_official_evaluation_and_visualization") as phase:
        suffix = f"A{phase.attempt:03d}"
        for key, group in search.groupby(group_columns, sort=True):
            profile, mode = key
            PROGRESS.emit("curve 계산 시작", compression_profile=profile, search_mode=mode)
            curve, metrics = official_curve(group, steps=CURVE_STEPS, rank=20)
            curve["compression_profile"] = profile
            curve["search_mode"] = mode
            curve_frames.append(curve)
            metric_rows.append({
                "compression_profile": profile,
                "search_mode": mode,
                "metric_label": "qmul-survface-v1-official-order-cosine-pgvector-adaptation",
                "distance_semantics": "pgvector_cosine_distance",
                "rank": 20,
                "official_complete": official_complete,
                **metrics,
            })

        metrics_frame = pd.DataFrame(metric_rows)
        curves_frame = pd.concat(curve_frames, ignore_index=True)
        metrics_path = phase.attempt_dir / f"official_metrics_{suffix}.csv"
        curves_path = phase.attempt_dir / f"tpir20_fpir_curves_{suffix}.csv"
        metrics_frame.to_csv(metrics_path, index=False, encoding="utf-8", lineterminator="\n")
        curves_frame.to_csv(curves_path, index=False, encoding="utf-8", lineterminator="\n")
        phase.publish_artifact(metrics_path)
        phase.publish_artifact(curves_path)

        fig, ax = plt.subplots(figsize=(7, 5))
        for (profile, mode), group in curves_frame.groupby(group_columns, sort=True):
            ax.plot(group["fpir"], group["tpir20"], label=f"{profile}/{mode}")
        for target in TARGET_FPIRS:
            ax.axvline(target, color="gray", linewidth=0.7, linestyle="--")
        ax.set(
            title="QMUL-SurvFace TPIR@FPIR (rank 20, cosine-pgvector adaptation)",
            xlabel="FPIR",
            ylabel="TPIR@rank20",
            xlim=(0.0, 1.0),
            ylim=(0.0, 1.0),
        )
        ax.legend()
        fig.tight_layout()
        figure_source = phase.attempt_dir / f"tpir20_fpir_curve_{suffix}.png"
        fig.savefig(figure_source, dpi=180)
        plt.close(fig)
        figure_destination = run.run_dir / "figures" / figure_source.name
        if figure_destination.exists():
            raise FileExistsError(figure_destination)
        os.replace(figure_source, figure_destination)
        phase.record(
            "figure_published",
            path=str(figure_destination.relative_to(run.run_dir)),
            sha256=sha256_file(figure_destination),
        )
        phase.record_counts(rows=len(search), metric_groups=len(metrics_frame), figures=1)

    if FINALIZE_RUN:
        run.complete()
    PROGRESS.emit("05 완료", rows=len(search), finalized=FINALIZE_RUN)
    result = {
        "status": "completed",
        "run_id": run.run_id,
        "source_search_csv": str(search_path),
        "official_complete": official_complete,
        "finalized": FINALIZE_RUN,
        "metrics": metric_rows,
    }
else:
    PROGRESS.emit("검토 모드 완료: 지표/그림/run 완료 처리를 실행하지 않음", expected="1초 미만")
result


## 최종 확인

표에는 `TPIR20@FPIR=0.1/0.2/0.3`, realized FPIR, AUC, rank-20 hit rate가 있어야 합니다. 결과 라벨의 `cosine-pgvector-adaptation`을 제거하지 마십시오. 공식 MATLAB Euclidean 재현 결과와 pgvector 적응 결과를 같은 열로 혼동하지 않고, exact/HNSW 및 compression profile을 분리해 보고합니다.
